# Hotel Reservation Chatbot — Improved NLP and Streamlit

This companion notebook demonstrates the same implementation used by `app.py`. The supplied original notebook is preserved. The hotel is fictional and reservations are stored locally; no real hotel booking or payment is made.

## Setup

Open this notebook from the extracted project folder. Install `requirements.txt` using the notebook's Python environment. See `README.md` for complete Streamlit setup commands. Python 3.12 is the tested runtime.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'app.py').exists():
    raise RuntimeError('Open the notebook with the working directory set to the extracted hotel-reservation-chatbot folder.')
sys.path.insert(0, str(ROOT))
from hotel_chatbot.config import load_config
from hotel_chatbot.intents import IntentRecognizer
from hotel_chatbot.reservations import ReservationService
from hotel_chatbot.conversation import HotelChatbot
config = load_config()
recognizer = IntentRecognizer()
print(config['name'])
print('Training examples:', len(recognizer.examples))


## NLP recognition

Explicit business requests use prioritized rules. Other messages use word/character TF-IDF and logistic regression. Rule scores express match strength; they are not calibrated probabilities. Unknown input falls back. Critical reservation changes require separate validation and explicit confirmation regardless of predicted intent.


In [ ]:
for message in ["I don't want a bot, I want a person", 'how much would that cost',
                'please cancel my booking', 'asdkj qweoi random gibberish']:
    prediction = recognizer.predict(message)
    print(message, '->', prediction.intent, prediction.source)


## A real local booking flow

The example below uses a disposable database. It shows context, accurate catalog pricing, identity collection, and explicit booking confirmation. The sample check-in is relative to the configured hotel timezone so the example stays runnable.


In [ ]:
from tempfile import TemporaryDirectory
from datetime import timedelta
scratch = TemporaryDirectory()
service = ReservationService(Path(scratch.name) / 'sample.sqlite3', config)
bot = HotelChatbot(service, recognizer)
checkin = service.today + timedelta(days=14)
messages = [f'Book a deluxe room for 3 nights starting {checkin.isoformat()} for 2 guests',
            'my name is Alex Rivera', 'alex@example.com', 'how much would that cost',
            'do you have wifi', 'confirm booking']
for message in messages:
    print('You:', message)
    print('Bot:', bot.respond(message), '\n')
print('Persisted local bookings:', service.count())


## Manage a reservation

Cancellation first verifies the reference and matching email, then asks for explicit confirmation. The demo does not process a payment or refund.


In [ ]:
reference = bot.last_booking['reference']
print(bot.respond(f'cancel booking {reference} alex@example.com'))
print(bot.respond('confirm cancellation'))
print('Status:', service.lookup(reference, 'alex@example.com')['status'])
scratch.cleanup()


## Reproduce evaluation

The original 96-message holdout has been inspected and informed rule coverage. Treat it as an exposed, tuned **regression benchmark**, not independent generalization evidence. The separate classifier-only result illustrates why the app uses hybrid routing and conservative validation. Human usability data and a fresh independent test set are still needed for real deployment.


In [ ]:
import subprocess
import json
subprocess.run([sys.executable, str(ROOT / 'scripts' / 'evaluate.py')], check=True)
report = json.loads((ROOT / 'evaluation' / 'results.json').read_text(encoding='utf-8'))
for key in ['classifier', 'hybrid']:
    result = report[key]
    print(key, f"{result['correct']}/{report['sample_count']}", f"accuracy {result['accuracy']:.2%}",
          f"macro-F1 {result['macro_f1']:.4f}", 'abstentions', result['abstentions'])
print(report['dataset_status'])


## Streamlit implementation

The complete interface is implemented in `app.py`: chat, conversation history, booking form, quote, confirmation controls, room catalog, receipt download, and reservation management. Run this command in a terminal from the project folder:

```bash
python -m streamlit run app.py
```

The next cell prints the command using the notebook's current Python interpreter. Starting Streamlit is intentionally a separate terminal action, so running notebook evaluation does not leave a background server running.


In [ ]:
import shlex
print(f'"{sys.executable}" -m streamlit run "{ROOT / "app.py"}"')
print('Run tests with:', f'"{sys.executable}" -m pytest')
